# 00_schema_and_folds

Этот ноутбук:
- стандартизирует входные данные;
- определяет `customer_id`, `target_cols`, `main_cat_cols`, `main_num_cols`, `extra_cols`;
- сохраняет схему в `prepared/artifacts/config`;
- строит **5-fold** разбиение через `MultilabelStratifiedKFold`;
- работает в Kaggle-стиле с бережным использованием RAM: читает метаданные без полной загрузки таблиц, таргеты — батчами, после крупных объектов выполняет очистку памяти.


In [1]:
!pip install iterative-stratification

In [2]:
# =========================
# Imports
# =========================
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


In [3]:
# =========================
# Paths
# =========================
DATA_DIR = Path("/kaggle/input/datasets/hatab123/data-fusion-contest-2026/")
WORK_DIR = Path("/kaggle/working/prepared")
WORK_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    "train_main": DATA_DIR / "train_main_features.parquet",
    "train_extra": DATA_DIR / "train_extra_features.parquet",
    "train_target": DATA_DIR / "train_target.parquet",
    "test_main": DATA_DIR / "test_main_features.parquet",
    "test_extra": DATA_DIR / "test_extra_features.parquet",
    "sample_submit": DATA_DIR / "sample_submit.parquet",
}

CONFIG_DIR = WORK_DIR / "artifacts" / "config"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

N_SPLITS = 5
RANDOM_STATE = 42
BATCH_SIZE = 100_000

print("DATA_DIR:", DATA_DIR)
print("WORK_DIR:", WORK_DIR)
for name, path in PATHS.items():
    print(f"{name:>14}: {path} | exists={path.exists()}")


DATA_DIR: /kaggle/input/datasets/hatab123/data-fusion-contest-2026
WORK_DIR: /kaggle/working/prepared
    train_main: /kaggle/input/datasets/hatab123/data-fusion-contest-2026/train_main_features.parquet | exists=True
   train_extra: /kaggle/input/datasets/hatab123/data-fusion-contest-2026/train_extra_features.parquet | exists=True
  train_target: /kaggle/input/datasets/hatab123/data-fusion-contest-2026/train_target.parquet | exists=True
     test_main: /kaggle/input/datasets/hatab123/data-fusion-contest-2026/test_main_features.parquet | exists=True
    test_extra: /kaggle/input/datasets/hatab123/data-fusion-contest-2026/test_extra_features.parquet | exists=True
 sample_submit: /kaggle/input/datasets/hatab123/data-fusion-contest-2026/sample_submit.parquet | exists=True


In [4]:
# =========================
# Helpers
# =========================
def parquet_info(path: Path):
    pf = pq.ParquetFile(path)
    schema_arrow = pf.schema_arrow
    cols = schema_arrow.names
    n_rows = pf.metadata.num_rows
    n_row_groups = pf.metadata.num_row_groups
    return pf, schema_arrow, cols, n_rows, n_row_groups


def detect_customer_id(columns):
    if "customer_id" in columns:
        return "customer_id"
    candidates = [c for c in columns if c.lower() == "customer_id" or c.lower().endswith("customer_id")]
    if len(candidates) == 1:
        return candidates[0]
    raise ValueError(f"Не удалось однозначно определить customer_id. Кандидаты: {candidates}")


def split_main_columns(schema_arrow, customer_id_col: str):
    cat_cols, num_cols, other_cols = [], [], []
    for field in schema_arrow:
        name = field.name
        if name == customer_id_col:
            continue

        low = name.lower()
        if low.startswith("cat_feature_"):
            cat_cols.append(name)
        elif low.startswith("num_feature_"):
            num_cols.append(name)
        else:
            if pa.types.is_string(field.type) or pa.types.is_large_string(field.type) or pa.types.is_dictionary(field.type):
                cat_cols.append(name)
            elif pa.types.is_integer(field.type) or pa.types.is_floating(field.type) or pa.types.is_boolean(field.type):
                num_cols.append(name)
            else:
                other_cols.append(name)
    return cat_cols, num_cols, other_cols


def read_target_matrix_batched(path: Path, customer_id_col: str, target_cols, batch_size=100_000):
    pf = pq.ParquetFile(path)
    n_rows = pf.metadata.num_rows
    n_targets = len(target_cols)

    y = np.empty((n_rows, n_targets), dtype=np.uint8)
    customer_ids = np.empty(n_rows, dtype=np.int64)

    start = 0
    for i, batch in enumerate(
        pf.iter_batches(columns=[customer_id_col] + target_cols, batch_size=batch_size), start=1
    ):
        batch_df = batch.to_pandas(types_mapper=pd.ArrowDtype)
        bs = len(batch_df)

        customer_ids[start:start+bs] = batch_df[customer_id_col].astype("int64").to_numpy(copy=False)
        y[start:start+bs] = (
            batch_df[target_cols]
            .astype("uint8", copy=False)
            .to_numpy(copy=True)
        )

        start += bs
        print(f"Read batch {i:02d}: rows={bs:,} | accumulated={start:,}/{n_rows:,}")

        del batch, batch_df, bs
        gc.collect()

    return customer_ids, y


def save_json(obj, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def summarize_schema(schema_arrow):
    rows = []
    for field in schema_arrow:
        rows.append(
            {
                "column": field.name,
                "arrow_type": str(field.type),
                "nullable": field.nullable,
            }
        )
    return pd.DataFrame(rows)


In [5]:
# =========================
# Read parquet metadata only
# =========================
train_main_pf, train_main_schema, train_main_cols, n_train_main, nrg_train_main = parquet_info(PATHS["train_main"])
train_extra_pf, train_extra_schema, train_extra_cols, n_train_extra, nrg_train_extra = parquet_info(PATHS["train_extra"])
train_target_pf, train_target_schema, train_target_cols_raw, n_train_target, nrg_train_target = parquet_info(PATHS["train_target"])
test_main_pf, test_main_schema, test_main_cols, n_test_main, nrg_test_main = parquet_info(PATHS["test_main"])
test_extra_pf, test_extra_schema, test_extra_cols, n_test_extra, nrg_test_extra = parquet_info(PATHS["test_extra"])

print("Rows:")
print(f"train_main  : {n_train_main:,}")
print(f"train_extra : {n_train_extra:,}")
print(f"train_target: {n_train_target:,}")
print(f"test_main   : {n_test_main:,}")
print(f"test_extra  : {n_test_extra:,}")

assert n_train_main == n_train_extra == n_train_target, "Train parquet files have different row counts"
assert n_test_main == n_test_extra, "Test parquet files have different row counts"


Rows:
train_main  : 750,000
train_extra : 750,000
train_target: 750,000
test_main   : 250,000
test_extra  : 250,000


In [6]:
train_main_schema

customer_id: int32
cat_feature_1: double
cat_feature_2: double
cat_feature_3: double
cat_feature_4: double
cat_feature_5: double
cat_feature_6: double
cat_feature_7: double
cat_feature_8: double
cat_feature_9: double
cat_feature_10: double
cat_feature_11: double
cat_feature_12: double
cat_feature_13: double
cat_feature_14: double
cat_feature_15: double
cat_feature_16: double
cat_feature_17: double
cat_feature_18: double
cat_feature_19: double
cat_feature_20: double
cat_feature_21: double
cat_feature_22: double
cat_feature_23: double
cat_feature_24: double
cat_feature_25: double
cat_feature_26: double
cat_feature_27: double
cat_feature_28: double
cat_feature_29: double
cat_feature_30: double
cat_feature_31: double
cat_feature_32: double
cat_feature_33: double
cat_feature_34: double
cat_feature_35: double
cat_feature_36: double
cat_feature_37: double
cat_feature_38: double
cat_feature_39: double
cat_feature_40: double
cat_feature_41: double
cat_feature_42: double
cat_feature_43: double
c

In [7]:
# =========================
# Detect key columns
# =========================
customer_id_col = detect_customer_id(train_main_cols)

assert customer_id_col in train_extra_cols, "customer_id not found in train_extra"
assert customer_id_col in train_target_cols_raw, "customer_id not found in train_target"
assert customer_id_col in test_main_cols, "customer_id not found in test_main"
assert customer_id_col in test_extra_cols, "customer_id not found in test_extra"

target_cols = [c for c in train_target_cols_raw if c != customer_id_col]
extra_cols = [c for c in train_extra_cols if c != customer_id_col]
main_cat_cols, main_num_cols, main_other_cols = split_main_columns(train_main_schema, customer_id_col)

print("customer_id_col :", customer_id_col)
print("targets         :", len(target_cols))
print("main_cat_cols   :", len(main_cat_cols))
print("main_num_cols   :", len(main_num_cols))
print("main_other_cols :", len(main_other_cols))
print("extra_cols      :", len(extra_cols))

if main_other_cols:
    print("\nColumns from train_main that did not match cat/num prefixes and were not clearly typed:")
    print(main_other_cols[:20], "..." if len(main_other_cols) > 20 else "")


customer_id_col : customer_id
targets         : 41
main_cat_cols   : 67
main_num_cols   : 132
main_other_cols : 0
extra_cols      : 2241


In [8]:
# =========================
# Sanity checks across train/test schemas
# =========================
test_main_non_id = [c for c in test_main_cols if c != customer_id_col]
test_extra_non_id = [c for c in test_extra_cols if c != customer_id_col]

assert set(main_cat_cols + main_num_cols + main_other_cols) == set(test_main_non_id), \
    "Mismatch between train_main and test_main columns"

assert set(extra_cols) == set(test_extra_non_id), \
    "Mismatch between train_extra and test_extra columns"

print("Schema checks passed.")


Schema checks passed.


In [9]:
# =========================
# Save schema files
# =========================
schema_payload = {
    "customer_id_col": customer_id_col,
    "n_splits": N_SPLITS,
    "random_state": RANDOM_STATE,
    "batch_size": BATCH_SIZE,
    "n_train_rows": int(n_train_target),
    "n_test_rows": int(n_test_main),
    "n_targets": len(target_cols),
    "n_main_cat_cols": len(main_cat_cols),
    "n_main_num_cols": len(main_num_cols),
    "n_main_other_cols": len(main_other_cols),
    "n_extra_cols": len(extra_cols),
    "paths": {k: str(v) for k, v in PATHS.items()},
}

save_json(schema_payload, CONFIG_DIR / "schema.json")
save_json(target_cols, CONFIG_DIR / "target_cols.json")
save_json(main_cat_cols, CONFIG_DIR / "main_cat_cols.json")
save_json(main_num_cols, CONFIG_DIR / "main_num_cols.json")
save_json(extra_cols, CONFIG_DIR / "extra_cols.json")

print("Saved:")
for fn in ["schema.json", "target_cols.json", "main_cat_cols.json", "main_num_cols.json", "extra_cols.json"]:
    print(" -", CONFIG_DIR / fn)


Saved:
 - /kaggle/working/prepared/artifacts/config/schema.json
 - /kaggle/working/prepared/artifacts/config/target_cols.json
 - /kaggle/working/prepared/artifacts/config/main_cat_cols.json
 - /kaggle/working/prepared/artifacts/config/main_num_cols.json
 - /kaggle/working/prepared/artifacts/config/extra_cols.json


In [10]:
# =========================
# Optional schema previews
# =========================
schema_main_preview = summarize_schema(train_main_schema)
schema_extra_preview = summarize_schema(train_extra_schema).head(10)
schema_target_preview = summarize_schema(train_target_schema)

display(schema_main_preview.head(20))
display(schema_extra_preview)
display(schema_target_preview.head(20))

del schema_main_preview, schema_extra_preview, schema_target_preview
gc.collect()


,column,arrow_type,nullable
0,customer_id,int32,True
1,cat_feature_1,double,True
2,cat_feature_2,double,True
3,cat_feature_3,double,True
4,cat_feature_4,double,True
5,cat_feature_5,double,True
6,cat_feature_6,double,True
7,cat_feature_7,double,True
8,cat_feature_8,double,True
9,cat_feature_9,double,True


,column,arrow_type,nullable
0,customer_id,int32,True
1,num_feature_133,double,True
2,num_feature_134,double,True
3,num_feature_135,double,True
4,num_feature_136,double,True
5,num_feature_137,double,True
6,num_feature_138,double,True
7,num_feature_139,double,True
8,num_feature_140,double,True
9,num_feature_141,double,True


,column,arrow_type,nullable
0,customer_id,int32,True
1,target_1_1,double,True
2,target_1_2,double,True
3,target_1_3,double,True
4,target_1_4,double,True
5,target_1_5,double,True
6,target_2_1,double,True
7,target_2_2,double,True
8,target_2_3,double,True
9,target_2_4,double,True


0

double — это тип данных из Arrow/Parquet schema, то есть аналог float64.

In [11]:
# =========================
# Read target matrix in batches
# =========================
customer_ids, y = read_target_matrix_batched(
    path=PATHS["train_target"],
    customer_id_col=customer_id_col,
    target_cols=target_cols,
    batch_size=BATCH_SIZE,
)

print("customer_ids shape:", customer_ids.shape)
print("y shape:", y.shape)
print("y dtype:", y.dtype)
print("mean positives per row:", float(y.sum(axis=1).mean()))


Read batch 01: rows=100,000 | accumulated=100,000/750,000
Read batch 02: rows=100,000 | accumulated=200,000/750,000
Read batch 03: rows=100,000 | accumulated=300,000/750,000
Read batch 04: rows=100,000 | accumulated=400,000/750,000
Read batch 05: rows=100,000 | accumulated=500,000/750,000
Read batch 06: rows=100,000 | accumulated=600,000/750,000
Read batch 07: rows=100,000 | accumulated=700,000/750,000
Read batch 08: rows=50,000 | accumulated=750,000/750,000
customer_ids shape: (750000,)
y shape: (750000, 41)
y dtype: uint8
mean positives per row: 1.3034586666666668


In [12]:
# =========================
# 5-fold multilabel stratified split
# =========================
mskf = MultilabelStratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

fold_assign = np.empty(len(customer_ids), dtype=np.int8)

for fold, (_, val_idx) in enumerate(mskf.split(np.zeros(len(customer_ids)), y)):
    fold_assign[val_idx] = fold
    print(f"Fold {fold}: {len(val_idx):,} rows")

folds_df = pd.DataFrame(
    {
        customer_id_col: customer_ids,
        "fold": fold_assign.astype("int8"),
    }
)

folds_path = CONFIG_DIR / "folds.parquet"
folds_df.to_parquet(folds_path, index=False)

print("Saved:", folds_path)
display(folds_df.head())
print(folds_df["fold"].value_counts().sort_index())


Fold 0: 149,895 rows
Fold 1: 150,030 rows
Fold 2: 150,017 rows
Fold 3: 150,124 rows
Fold 4: 149,934 rows
Saved: /kaggle/working/prepared/artifacts/config/folds.parquet


,customer_id,fold
0,1000001,4
1,1000002,4
2,1000003,4
3,1000004,1
4,1000005,1


fold
0    149895
1    150030
2    150017
3    150124
4    149934
Name: count, dtype: int64


In [13]:
# =========================
# Fold quality checks
# =========================
global_mean = y.mean(axis=0)

rows = []
fold_values = folds_df["fold"].to_numpy()
for fold in range(N_SPLITS):
    mask = fold_values == fold
    fold_mean = y[mask].mean(axis=0)
    abs_diff = np.abs(fold_mean - global_mean)
    rows.append(
        {
            "fold": fold,
            "rows": int(mask.sum()),
            "macro_target_rate_abs_diff_mean": float(abs_diff.mean()),
            "macro_target_rate_abs_diff_max": float(abs_diff.max()),
            "avg_targets_per_row": float(y[mask].sum(axis=1).mean()),
        }
    )

fold_quality = pd.DataFrame(rows)
display(fold_quality)


,fold,rows,macro_target_rate_abs_diff_mean,macro_target_rate_abs_diff_max,avg_targets_per_row
0,0,149895,0.000023,0.000222,1.304366
1,1,150030,0.000008,0.000062,1.303139
2,2,150017,0.000005,0.000034,1.303339
3,3,150124,0.000027,0.000266,1.302370
4,4,149934,0.000016,0.000140,1.304080


In [14]:
# =========================
# Final cleanup
# =========================
del customer_ids, y, fold_assign, folds_df, global_mean, rows, fold_quality, fold_values
gc.collect()

print("Done. Config artifacts are ready in:")
print(CONFIG_DIR)


Done. Config artifacts are ready in:
/kaggle/working/prepared/artifacts/config
